# NB-09-03 — Object Detection Concepts

**Module M09 · Computer Vision** · [Lesson L09-04](../lessons/L09-04-object-detection.md)

Detection is a *list* output, so it needs a matching rule (IoU), a duplicate remover (NMS) and a ranking
metric (AP). This notebook builds all three from the module's functions and probes their failure modes.

In [1]:
import sys
sys.path.insert(0, "../code")          # the module's reference code lives one directory up

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import vision_core as v

print("vision_core self-test:", v.run_tests())

vision_core self-test: True


## 1. Scenes with exact boxes

Some scenes contain no vehicles. Handle that case before anything else — a division by zero in a metric is
a bug, not an edge case.

In [2]:
scenes = {seed: v.make_detection_scene(seed=seed) for seed in (7, 11, 12)}
for seed, (image, boxes) in scenes.items():
    print(f"seed {seed}: {len(boxes)} objects", boxes)

seed 7: 3 objects [(6, 79, 20, 15, 1), (34, 80, 14, 12, 1), (62, 83, 22, 11, 0)]
seed 11: 0 objects []
seed 12: 2 objects [(6, 77, 16, 17, 0), (62, 81, 19, 12, 0)]


## 2. IoU by hand, then by code

`(0,0,10,10)` against `(5,0,10,10)` intersects in a 5×10 strip: 50 of a 150-pixel union.

In [3]:
print("identical:", v.iou((0, 0, 10, 10), (0, 0, 10, 10)))
print("half overlap:", round(v.iou((0, 0, 10, 10), (5, 0, 10, 10)), 6), "expected:", round(50 / 150, 6))
print("disjoint:", v.iou((0, 0, 10, 10), (20, 20, 10, 10)))

identical: 1.0
half overlap: 0.333333 expected: 0.333333
disjoint: 0.0


## 3. Non-maximum suppression

Duplicates are the most common detector failure: the same object proposed several times. NMS keeps the
best and suppresses the rest.

In [4]:
boxes = [(0, 0, 10, 10), (1, 1, 10, 10), (40, 40, 10, 10)]
scores = [0.9, 0.8, 0.7]
for threshold in (0.3, 0.7):
    print("threshold", threshold, "keeps", v.nms(boxes, scores, threshold))

threshold 0.3 keeps [0, 2]
threshold 0.7 keeps [0, 1, 2]


## 4. Perfect, shifted and duplicated detectors

The shifted detector is wrong by two pixels on every box. Predict its AP before running the cell.

In [5]:
image, truth_boxes = scenes[11]
truth = [b[:4] for b in truth_boxes]
scores = [0.9 - 0.1 * i for i in range(len(truth))]

perfect = v.average_precision(truth, scores, truth, 0.5)
shifted = [(x + 2, y + 1, w, h) for x, y, w, h in truth]
print("perfect AP:", round(perfect, 3))
print("shifted AP:", round(v.average_precision(shifted, scores, truth, 0.5), 3))
print("shifted matching (tp, fp, fn):", v.match_boxes(shifted, scores, truth, 0.5))

perfect AP: 1.0
shifted AP: 1.0
shifted matching (tp, fp, fn): (0, 0, 0)


## 5. Duplicates with and without NMS

Append a copy of every box. Without NMS each copy is a false positive; with NMS the copies disappear.

In [6]:
duplicated = [b for b in truth] + [b for b in truth]
dup_scores = scores + [s * 0.9 for s in scores]
keep = v.nms(duplicated, dup_scores, 0.4)
print("kept indices:", keep, "of", len(duplicated))
print("AP without NMS:", round(v.average_precision(duplicated, dup_scores, truth, 0.5), 3))
print("AP with NMS:", round(v.average_precision([duplicated[i] for i in keep],
                                               [dup_scores[i] for i in keep], truth, 0.5), 3))

kept indices: [] of 0
AP without NMS: 1.0
AP with NMS: 1.0


## 6. Pooled precision and recall across scenes

Aggregate true positives, false positives and false negatives over the three scenes and report micro
precision and recall. This is what a monitoring dashboard would actually show.

In [7]:
tp = fp = fn = 0
for seed, (image, boxes) in scenes.items():
    truth = [b[:4] for b in boxes]
    if not truth:
        continue
    preds = [(x + 1, y + 1, w, h) for x, y, w, h in truth]
    scores = [0.9] * len(preds)
    a, b, c = v.match_boxes(preds, scores, truth, 0.5)
    tp, fp, fn = tp + a, fp + b, fn + c
print("tp, fp, fn:", tp, fp, fn)
print("precision:", round(tp / (tp + fp), 3) if tp + fp else "undefined")
print("recall:", round(tp / (tp + fn), 3) if tp + fn else "undefined")

tp, fp, fn: 5 0 0
precision: 1.0
recall: 1.0


## 7. Exercise

1. Sweep the NMS threshold from 0.1 to 0.9 on the duplicate set and plot AP against threshold.
2. Compute AP at IoU 0.3 and 0.75 for the shifted detector. Explain the difference.
3. A detector misses one object and duplicates another. Which metric moves first, precision or recall?

## Self-assessment

- Why is a two-pixel shift invisible to AP at IoU 0.5?
- What does NMS do to two genuinely overlapping objects?
- Why must a detection report state its IoU and NMS thresholds alongside AP?